# **2일차 팀 프로젝트: 카드 사용 기록 조회·분석 RAG 챗봇**

## 프로젝트 목표
1. 카드 기록 에셋 10개를 모두 구조화하여 Qdrant Cloud에 저장
2. 날짜·카드사·가맹점·금액·카테고리를 바탕으로 관련 거래 조회
3. pandas의 정확한 집계와 RAG 검색 결과를 결합하여 소비 패턴 분석
4. 답변마다 근거 거래 또는 집계 기준을 함께 제시

## 사용 데이터
- `*_usage.xlsx` 6개: 카드 이용내역
- `이용대금명세서(신용카드).xls` 3개: HTML 형식의 신한카드 명세서
- `팝업 _ 카드 이용내역 - 삼성카드.pdf` 1개: 삼성카드 이용내역 확인서
- 총 10개 에셋을 모두 사용합니다.

카드사별 거래는 서로 다른 실제 결제로 취급합니다. 카드번호와 주민등록번호는 로딩 과정에서
사용하지 않으며 Document, 집계 데이터, Qdrant에도 저장하지 않습니다.

> 이 프로젝트는 00~05 예제의 환경 설정, Document, Qdrant, 메타데이터 필터링,
> Retriever, PromptTemplate 패턴을 카드 기록 도메인에 맞게 결합했습니다.


## 0. 환경 변수 설정

In [84]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

required_env = ["OPENAI_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"]
missing_env = [name for name in required_env if not os.getenv(name)]

if missing_env:
    raise EnvironmentError(f".env에 다음 환경 변수를 설정하세요: {', '.join(missing_env)}")

print("✓ OpenAI API Key가 설정되었습니다.")
print("✓ Qdrant Cloud 설정이 완료되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")


✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://5b074511-eff4-41d6-8470-919b0d5292af.us-east-2-0.aws.cloud.qdrant.io:6333


## 1. 모든 카드 사용 기록 로딩

파일 형식별 전용 로더를 사용합니다.

- XLSX: XML을 직접 읽어 추가 Excel 엔진 없이 처리
- XLS: 실제 내용이 HTML이므로 BeautifulSoup으로 거래 표 추출
- PDF: PyMuPDF의 단어 좌표로 줄바꿈된 가맹점명까지 거래 행 단위로 복원

모든 결과는 `이용일`, `카드사`, `가맹점`, `금액`, `원본파일`, `원본행` 구조로 통합합니다.


In [85]:
import re
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import fitz
from bs4 import BeautifulSoup

DATA_DIR = Path("../datasets/카드사용기록")
xlsx_files = sorted(DATA_DIR.glob("*_usage.xlsx"))
xls_files = sorted(DATA_DIR.glob("*.xls"))
pdf_files = sorted(DATA_DIR.glob("*.pdf"))
asset_files = xlsx_files + xls_files + pdf_files

if not asset_files:
    raise FileNotFoundError(f"카드 기록 파일을 찾을 수 없습니다: {DATA_DIR.resolve()}")


def parse_money(value: str) -> int:
    """쉼표와 원 표시가 포함된 금액 문자열을 정수로 변환합니다."""
    cleaned = str(value).replace(",", "").replace("원", "").strip()
    return int(float(cleaned)) if cleaned not in {"", "-"} else 0


def read_usage_xlsx(file_path: Path) -> pd.DataFrame:
    """카드 이용내역 XLSX의 거래 행을 읽습니다."""
    namespace = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(file_path) as workbook:
        sheet = ET.fromstring(workbook.read("xl/worksheets/sheet1.xml"))

    rows = []
    for row_number, row in enumerate(
        sheet.findall(".//m:sheetData/m:row", namespace), start=1
    ):
        values = {}
        for cell in row.findall("m:c", namespace):
            column = re.match(r"[A-Z]+", cell.attrib["r"]).group()
            inline_text = "".join(
                node.text or "" for node in cell.findall(".//m:t", namespace)
            )
            value_node = cell.find("m:v", namespace)
            values[column] = (
                inline_text if inline_text else value_node.text if value_node is not None else ""
            )

        usage_date = values.get("C", "").replace("\u00a0", "").strip()
        if not re.fullmatch(r"\d{4}\.\d{2}\.\d{2}", usage_date):
            continue

        rows.append(
            {
                "이용일": usage_date.replace(".", "-"),
                "카드사": "KB",
                "가맹점": values.get("E", "").replace("\u00a0", "").strip(),
                "금액": parse_money(values.get("F", "0")),
                "원본파일": file_path.name,
                "원본행": f"xlsx-{row_number}",
            }
        )
    return pd.DataFrame(rows)


def read_statement_xls(file_path: Path) -> pd.DataFrame:
    """HTML 형식으로 저장된 신한카드 XLS 명세서의 거래 표를 읽습니다."""
    soup = BeautifulSoup(file_path.read_text(encoding="utf-8"), "html.parser")
    candidates = []

    for table in soup.find_all("table"):
        table_rows = []
        for tr in table.find_all("tr"):
            cells = [
                " ".join(cell.get_text(" ", strip=True).split())
                for cell in tr.find_all(["th", "td"], recursive=False)
            ]
            if cells:
                table_rows.append(cells)

        has_usage_header = any(
            len(row) >= 4
            and row[:4] == ["이용일", "이용카드", "이용가맹점", "이용금액"]
            for row in table_rows
        )
        if has_usage_header:
            candidates.append((len(table.find_all("table")), table_rows))

    if not candidates:
        raise ValueError(f"카드사용내역 표를 찾을 수 없습니다: {file_path.name}")

    # 중첩된 바깥 표가 아닌 가장 안쪽 거래 표를 사용합니다.
    _, transaction_rows = min(candidates, key=lambda item: item[0])
    rows = []
    for row_number, values in enumerate(transaction_rows, start=1):
        if len(values) < 7 or not re.fullmatch(r"\d{4}\.\d{2}\.\d{2}", values[0]):
            continue

        foreign_fee = parse_money(values[7]) if len(values) > 7 else 0
        # 해외 이용 건은 이용금액이 외화이므로 이번 달 납부 원금을 원화 금액으로 사용합니다.
        amount = parse_money(values[6]) if foreign_fee else parse_money(values[3])
        rows.append(
            {
                "이용일": values[0].replace(".", "-"),
                "카드사": "신한",
                "가맹점": values[2],
                "금액": amount,
                "원본파일": file_path.name,
                "원본행": f"xls-{row_number}",
            }
        )
    return pd.DataFrame(rows)


def read_samsung_pdf(file_path: Path) -> pd.DataFrame:
    """삼성카드 PDF 표를 단어 좌표로 분석해 거래 행을 읽습니다."""
    document = fitz.open(file_path)
    rows = []

    try:
        for page_number, page in enumerate(document, start=1):
            words = page.get_text("words", sort=True)
            date_words = sorted(
                [
                    word for word in words
                    if 140 <= word[0] <= 225
                    and re.fullmatch(r"\d{4}\.\d{2}\.\d{2}", word[4])
                ],
                key=lambda word: word[1],
            )

            date_centers = [(word[1] + word[3]) / 2 for word in date_words]
            for index, date_word in enumerate(date_words):
                center = date_centers[index]
                lower = (
                    (date_centers[index - 1] + center) / 2
                    if index > 0 else center - 25
                )
                upper = (
                    (center + date_centers[index + 1]) / 2
                    if index + 1 < len(date_centers) else center + 25
                )

                merchant_tokens = [
                    word for word in words
                    if 225 <= word[0] < 345
                    and lower <= (word[1] + word[3]) / 2 < upper
                ]
                amount_tokens = [
                    word for word in words
                    if 430 <= word[0] < 525
                    and lower <= (word[1] + word[3]) / 2 < upper
                    and re.fullmatch(r"-?[\d,]+원", word[4])
                ]
                approval_tokens = [
                    word for word in words
                    if 345 <= word[0] < 425
                    and lower <= (word[1] + word[3]) / 2 < upper
                    and re.fullmatch(r"\d{8}", word[4])
                ]

                if not merchant_tokens or not amount_tokens:
                    continue

                merchant_tokens.sort(key=lambda word: (round(word[1], 1), word[0]))
                merchant = " ".join(word[4] for word in merchant_tokens).strip()
                approval = approval_tokens[0][4] if approval_tokens else str(index + 1)
                rows.append(
                    {
                        "이용일": date_word[4].replace(".", "-"),
                        "카드사": "삼성",
                        "가맹점": merchant,
                        "금액": parse_money(amount_tokens[0][4]),
                        "원본파일": file_path.name,
                        "원본행": f"pdf-{page_number}-{approval}-{index + 1}",
                    }
                )
    finally:
        document.close()

    return pd.DataFrame(rows)


source_frames = []
for file_path in xlsx_files:
    source_frames.append(read_usage_xlsx(file_path))
for file_path in xls_files:
    source_frames.append(read_statement_xls(file_path))
for file_path in pdf_files:
    source_frames.append(read_samsung_pdf(file_path))

transactions = pd.concat(source_frames, ignore_index=True)
transactions["이용일"] = pd.to_datetime(transactions["이용일"], format="%Y-%m-%d")
transactions["월"] = transactions["이용일"].dt.strftime("%Y-%m")
transactions["거래유형"] = transactions["금액"].map(lambda amount: "취소" if amount < 0 else "승인")

print(f"✓ 카드 데이터 에셋 {len(asset_files)}개 전체 로드 완료")
print(f"  - XLSX: {len(xlsx_files)}개")
print(f"  - HTML XLS: {len(xls_files)}개")
print(f"  - PDF: {len(pdf_files)}개")
print(f"✓ 거래 {len(transactions):,}건 ({transactions['이용일'].min():%Y-%m-%d} ~ {transactions['이용일'].max():%Y-%m-%d})")
print(f"✓ 전체 순사용액: {transactions['금액'].sum():,}원")

source_summary = (
    transactions.groupby(["카드사", "원본파일"], as_index=False)
    .agg(거래건수=("금액", "size"), 순사용액=("금액", "sum"))
)
display(source_summary)


✓ 카드 데이터 에셋 10개 전체 로드 완료
  - XLSX: 6개
  - HTML XLS: 3개
  - PDF: 1개
✓ 거래 496건 (2026-02-01 ~ 2026-08-19)
✓ 전체 순사용액: 9,759,902원


,카드사,원본파일,거래건수,순사용액
0,KB,202602_usage.xlsx,18,341703
1,KB,202603_usage.xlsx,27,405689
2,KB,202604_usage.xlsx,36,776133
3,KB,202605_usage.xlsx,31,245640
4,KB,202606_usage.xlsx,45,330443
5,KB,202607_usage.xlsx,30,298042
6,삼성,팝업 _ 카드 이용내역 - 삼성카드.pdf,185,3053709
7,신한,2026년6월 이용대금명세서(신용카드).xls,55,2283380
8,신한,2026년7월 이용대금명세서(신용카드).xls,33,1313940
9,신한,2026년8월 이용대금명세서(신용카드).xls,36,711223


## 2. 거래 카테고리 분류 및 Document 생성

거래 1건을 하나의 Document로 만듭니다. 카드 거래는 이미 짧고 구조화되어 있으므로
문자 수 기준 청킹보다 거래 단위를 보존하는 것이 검색과 금액 근거 제시에 적합합니다.

카드번호 대신 카드사·원본파일·원본행을 조합한 거래 ID를 사용합니다.


In [86]:
from langchain_core.documents import Document


def classify_category(merchant: str) -> str:
    """가맹점명 키워드로 소비 카테고리를 분류합니다."""
    rules = {
        "식비": ["식당", "갈비", "리아", "배달", "우아한", "쿠팡이츠", "김밥", "치킨", "푸드", "버거", "분식", "국밥", "쌀국수"],
        "카페·간식": ["커피", "카페", "아이스크림", "베이커리", "메가MGC", "스타벅스", "설빙", "와플", "던킨"],
        "편의점·마트": ["CU", "씨유", "GS25", "지에스25", "세븐일레븐", "SEVEN-ELEVEN", "마트", "슈퍼", "컬리", "쿠팡(쿠페이)"],
        "교통·차량": ["피플카", "지하철", "택시", "철도", "코레일", "버스", "주유", "나눔에너지", "톨게이트", "통행료", "주차", "아이파킹"],
        "주거·공과금": ["전기요금", "통신요금", "도시가스", "관리비"],
        "의료·보험": ["보험", "약국", "의원", "병원", "치과"],
        "쇼핑·생활": ["빨래방", "다이소", "올리브영", "백화점", "아디다스", "한섬", "유니클로", "에프알엘"],
        "교육·디지털": ["데이터산업진흥원", "Google", "GOOGLE", "APPLE", "구글", "네이버", "넥슨", "정보인증"],
        "여가": ["메가박스", "롯데월드", "보드카페", "드림월드"],
    }
    lowered = merchant.lower()
    for category, keywords in rules.items():
        if any(keyword.lower() in lowered for keyword in keywords):
            return category
    return "기타"


transactions["카테고리"] = transactions["가맹점"].map(classify_category)
transactions["거래ID"] = transactions.apply(
    lambda row: f"{row['원본파일']}:{row['원본행']}",
    axis=1,
)

docs = []
for _, row in transactions.iterrows():
    usage_date = row["이용일"].strftime("%Y-%m-%d")
    content = (
        f"카드 거래 기록 | 카드사: {row['카드사']} | 이용일: {usage_date} | "
        f"가맹점: {row['가맹점']} | 금액: {row['금액']:,}원 | "
        f"거래유형: {row['거래유형']} | 카테고리: {row['카테고리']}"
    )
    docs.append(
        Document(
            page_content=content,
            metadata={
                "transaction_id": row["거래ID"],
                "source": row["원본파일"],
                "source_row": row["원본행"],
                "card_company": row["카드사"],
                "date": usage_date,
                "month": row["월"],
                "merchant": row["가맹점"],
                "amount": int(row["금액"]),
                "transaction_type": row["거래유형"],
                "category": row["카테고리"],
            },
        )
    )

print(f"✓ {len(docs):,}개의 거래 Document 생성 완료")
print(f"\n예시:\n{docs[0].page_content}")
print(f"메타데이터: {docs[0].metadata}")


✓ 496개의 거래 Document 생성 완료

예시:
카드 거래 기록 | 카드사: KB | 이용일: 2026-02-01 | 가맹점: 쿠팡(쿠페이)-쿠팡(쿠페이) | 금액: 221,540원 | 거래유형: 승인 | 카테고리: 편의점·마트
메타데이터: {'transaction_id': '202602_usage.xlsx:xlsx-2', 'source': '202602_usage.xlsx', 'source_row': 'xlsx-2', 'card_company': 'KB', 'date': '2026-02-01', 'month': '2026-02', 'merchant': '쿠팡(쿠페이)-쿠팡(쿠페이)', 'amount': 221540, 'transaction_type': '승인', 'category': '편의점·마트'}


## 3. Qdrant Cloud에 카드 거래 저장

- 임베딩 모델은 03~05 예제와 동일한 `text-embedding-3-large`를 사용합니다.
- 거래 ID로 만든 고정 UUID를 사용하므로 셀을 다시 실행해도 중복 포인트가 생기지 않습니다.
- 큰 벡터의 네트워크 쓰기 타임아웃을 줄이기 위해 `timeout=120`, `batch_size=16`을 사용합니다.


In [87]:
from uuid import NAMESPACE_URL, uuid5
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, PayloadSchemaType, VectorParams

client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    timeout=120,
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
collection_name = "card_usage_records_all_assets"

existing_names = {collection.name for collection in client.get_collections().collections}
if collection_name not in existing_names:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
    )
    print(f"✓ 컬렉션 '{collection_name}' 생성 완료")
else:
    print(f"✓ 기존 컬렉션 '{collection_name}'을 사용합니다.")

# 동적 메타데이터 필터에 필요한 keyword 인덱스 생성
filter_fields = [
    "metadata.month",
    "metadata.card_company",
    "metadata.category",
    "metadata.transaction_type",
]
collection_info = client.get_collection(collection_name)
for field_name in filter_fields:
    if field_name not in collection_info.payload_schema:
        client.create_payload_index(
            collection_name=collection_name,
            field_name=field_name,
            field_schema=PayloadSchemaType.KEYWORD,
            wait=True,
        )
        print(f"✓ {field_name} keyword 인덱스 생성 완료")
    else:
        print(f"✓ {field_name} keyword 인덱스가 이미 존재합니다.")

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)

document_ids = [str(uuid5(NAMESPACE_URL, doc.metadata["transaction_id"])) for doc in docs]
vectorstore.add_documents(documents=docs, ids=document_ids, batch_size=16)

point_count = client.get_collection(collection_name).points_count
print(f"✓ 카드 거래 {len(docs):,}건 업서트 완료 (현재 포인트: {point_count:,}개)")


✓ 기존 컬렉션 'card_usage_records_all_assets'을 사용합니다.
✓ metadata.month keyword 인덱스가 이미 존재합니다.
✓ metadata.card_company keyword 인덱스가 이미 존재합니다.
✓ metadata.category keyword 인덱스가 이미 존재합니다.
✓ metadata.transaction_type keyword 인덱스가 이미 존재합니다.
✓ 카드 거래 496건 업서트 완료 (현재 포인트: 496개)


## 4. 카드 기록 Retriever 구현

질문에서 다음 조건을 자동으로 추출해 Qdrant 메타데이터 필터에 적용합니다.

- 월: `7월`, `2026년 7월`
- 카드사: `KB`, `신한`, `삼성`
- 카테고리: 식비, 카페, 교통, 의료 등
- 거래유형: 승인, 취소/환불

의미 검색과 구조화 필터를 결합해 관련 없는 카드 기록이 섞이는 것을 줄입니다.


In [88]:
from typing import Dict, List, Optional
from qdrant_client import models


CATEGORY_ALIASES = {
    "식비": ["식비", "식사", "음식", "배달"],
    "카페·간식": ["카페", "커피", "간식", "디저트"],
    "편의점·마트": ["편의점", "마트", "장보기"],
    "교통·차량": ["교통", "차량", "택시", "버스", "지하철", "주유", "주차"],
    "주거·공과금": ["공과금", "주거", "전기", "통신요금"],
    "의료·보험": ["의료", "병원", "약국", "보험"],
    "쇼핑·생활": ["쇼핑", "생활", "의류"],
    "교육·디지털": ["교육", "디지털", "게임", "구독"],
    "여가": ["여가", "영화", "놀이"],
}


def extract_month(question: str) -> Optional[str]:
    """질문에서 2026년의 월을 YYYY-MM 형식으로 추출합니다."""
    match = re.search(r"(?:(20\d{2})년\s*)?(1[0-2]|0?[1-9])월", question)
    if not match:
        return None
    year = match.group(1) or "2026"
    return f"{year}-{int(match.group(2)):02d}"


def extract_month_range(question: str):
    """질문에서 '2월부터 7월까지'와 같은 월 범위를 추출합니다."""
    match = re.search(
        r"(?:(20\d{2})년\s*)?(1[0-2]|0?[1-9])월\s*"
        r"(?:부터|에서|~|-)\s*"
        r"(?:(20\d{2})년\s*)?(1[0-2]|0?[1-9])월(?:까지)?",
        question,
    )
    if not match:
        return None
    start_year = match.group(1) or "2026"
    end_year = match.group(3) or start_year
    start = f"{start_year}-{int(match.group(2)):02d}"
    end = f"{end_year}-{int(match.group(4)):02d}"
    return (start, end) if start <= end else (end, start)


def months_between(start: str, end: str) -> List[str]:
    """두 YYYY-MM 사이의 모든 월을 반환합니다."""
    return [str(period) for period in pd.period_range(start, end, freq="M")]


def extract_query_filters(question: str) -> Dict[str, object]:
    """질문에서 월·카드사·카테고리·거래유형 필터를 추출합니다."""
    filters = {}
    month_range = extract_month_range(question)
    if month_range:
        filters["month_range"] = month_range
    else:
        month = extract_month(question)
        if month:
            filters["month"] = month

    card_aliases = {
        "KB": ["KB", "국민"],
        "신한": ["신한"],
        "삼성": ["삼성"],
    }
    lowered = question.lower()
    for card_company, aliases in card_aliases.items():
        if any(alias.lower() in lowered for alias in aliases):
            filters["card_company"] = card_company
            break

    for category, aliases in CATEGORY_ALIASES.items():
        if any(alias in question for alias in aliases):
            filters["category"] = category
            break

    if any(word in question for word in ["취소", "환불", "마이너스"]):
        filters["transaction_type"] = "취소"
    elif "승인" in question:
        filters["transaction_type"] = "승인"

    return filters


class CardRecordRetriever:
    def __init__(self, vectorstore: QdrantVectorStore, k: int = 12):
        self.vectorstore = vectorstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        filters = extract_query_filters(query)
        conditions = []
        for field_name, value in filters.items():
            if field_name == "month_range":
                conditions.append(
                    models.FieldCondition(
                        key="metadata.month",
                        match=models.MatchAny(any=months_between(*value)),
                    )
                )
            else:
                conditions.append(
                    models.FieldCondition(
                        key=f"metadata.{field_name}",
                        match=models.MatchValue(value=value),
                    )
                )
        query_filter = models.Filter(must=conditions) if conditions else None
        return self.vectorstore.similarity_search(
            query,
            k=self.k,
            filter=query_filter,
        )


card_retriever = CardRecordRetriever(vectorstore=vectorstore, k=12)
print("✓ 카드 기록 Retriever 생성 완료")


✓ 카드 기록 Retriever 생성 완료


## 5. 검색 테스트

In [89]:
query = "삼성카드 7월 교통비 기록을 보여줘"
search_results = card_retriever.invoke(query)

print(f"검색 질문: {query}")
print(f"적용 필터: {extract_query_filters(query) or '없음'}")
print(f"검색 결과: {len(search_results)}건\n")

for index, result in enumerate(search_results, start=1):
    metadata = result.metadata
    print(
        f"{index}. {metadata['date']} | {metadata['card_company']} | "
        f"{metadata['merchant']} | {metadata['amount']:,}원 | "
        f"{metadata['transaction_type']}"
    )


검색 질문: 삼성카드 7월 교통비 기록을 보여줘
적용 필터: {'month': '2026-07', 'card_company': '삼성', 'category': '교통·차량'}
검색 결과: 4건

1. 2026-07-11 | 삼성 | 코레일유통(주) | 1,100원 | 승인
2. 2026-07-25 | 삼성 | 코레일유통(주) | 1,100원 | 승인
3. 2026-07-29 | 삼성 | 코레일유통(주) | 1,100원 | 승인
4. 2026-07-01 | 삼성 | 티머니택시(법인)_4 | 9,900원 | 승인


## 6. 정확한 소비 분석 컨텍스트 생성

RAG 검색은 관련 거래를 찾는 데 강하지만 모든 행의 합계를 계산하는 용도에는 적합하지 않습니다.
따라서 pandas가 전체 카드 기록을 정확히 집계하고, LLM에는 그 결과를 근거로 제공합니다.


In [90]:
def make_summary_tables(data: pd.DataFrame, title: str = "전체") -> str:
    """선택된 카드 기록의 정확한 집계표를 텍스트로 만듭니다."""
    if data.empty:
        return f"[{title} 집계]\n조건에 해당하는 거래가 없습니다."

    approvals = data.loc[data["금액"] > 0, "금액"].sum()
    refunds = data.loc[data["금액"] < 0, "금액"].sum()
    monthly = (
        data.groupby("월", as_index=False)
        .agg(거래건수=("금액", "size"), 순사용액=("금액", "sum"))
        .sort_values("월")
    )
    card_company = (
        data.groupby("카드사", as_index=False)
        .agg(거래건수=("금액", "size"), 순사용액=("금액", "sum"))
        .sort_values("순사용액", ascending=False)
    )
    category = (
        data.groupby("카테고리", as_index=False)
        .agg(거래건수=("금액", "size"), 순사용액=("금액", "sum"))
        .sort_values("순사용액", ascending=False)
    )
    merchant = (
        data.groupby("가맹점", as_index=False)
        .agg(거래건수=("금액", "size"), 순사용액=("금액", "sum"))
        .sort_values("순사용액", ascending=False)
        .head(15)
    )

    return "\n\n".join(
        [
            f"[{title} 요약]\n"
            f"기간: {data['이용일'].min():%Y-%m-%d} ~ {data['이용일'].max():%Y-%m-%d}\n"
            f"거래 건수: {len(data):,}건\n"
            f"승인 합계: {approvals:,}원\n"
            f"취소 합계: {refunds:,}원\n"
            f"순사용액(승인+취소): {data['금액'].sum():,}원",
            "[월별 집계]\n" + monthly.to_string(index=False),
            "[카드사별 집계]\n" + card_company.to_string(index=False),
            "[카테고리별 집계]\n" + category.to_string(index=False),
            "[순사용액 상위 가맹점 15개]\n" + merchant.to_string(index=False),
        ]
    )


def _strip_particle(token: str) -> str:
    for suffix in ["에서", "으로", "에게", "한테", "의", "은", "는", "이", "가", "을", "를", "로"]:
        if token.endswith(suffix) and len(token) > len(suffix) + 1:
            return token[:-len(suffix)]
    return token


def filter_transactions_for_question(question: str):
    """질문의 구조화 조건과 가맹점 표현으로 pandas 원장을 정확히 필터링합니다."""
    data = transactions.copy()
    filters = extract_query_filters(question)
    column_map = {
        "month": "월",
        "card_company": "카드사",
        "category": "카테고리",
        "transaction_type": "거래유형",
    }
    applied = []
    for field_name, value in filters.items():
        if field_name == "month_range":
            start, end = value
            data = data[data["월"].between(start, end)]
            applied.append(f"월={start}~{end}")
        else:
            data = data[data[column_map[field_name]] == value]
            applied.append(f"{column_map[field_name]}={value}")

    stop_words = {
        "카드", "기록", "내역", "거래", "결제", "사용", "사용한", "쓴", "돈",
        "보여줘", "알려줘", "찾아줘", "분석", "비교", "전체", "합계", "얼마",
        "승인", "취소", "환불", "월별", "카테고리별", "가맹점별",
        "kb", "국민", "신한", "삼성",
    }
    stop_words.update(
        alias.lower()
        for aliases in CATEGORY_ALIASES.values()
        for alias in aliases
    )
    tokens = [
        _strip_particle(token.lower())
        for token in re.findall(r"[가-힣A-Za-z0-9()·]+", question)
    ]
    merchant_tokens = []
    merchants_lower = transactions["가맹점"].str.lower()
    for token in tokens:
        if len(token) < 2 or token in stop_words or re.fullmatch(r"\d+월?", token):
            continue
        if merchants_lower.str.contains(re.escape(token), regex=True).any():
            merchant_tokens.append(token)

    if merchant_tokens:
        merchant_mask = data["가맹점"].str.lower().apply(
            lambda merchant: any(token in merchant for token in merchant_tokens)
        )
        data = data[merchant_mask]
        applied.append(f"가맹점 키워드={','.join(merchant_tokens)}")

    return data, applied


analysis_summary = make_summary_tables(transactions)
print(analysis_summary)


[전체 요약]
기간: 2026-02-01 ~ 2026-08-19
거래 건수: 496건
승인 합계: 10,015,921원
취소 합계: -256,019원
순사용액(승인+취소): 9,759,902원

[월별 집계]
      월  거래건수    순사용액
2026-02    18  341703
2026-03    27  405689
2026-04    46 1025733
2026-05   126 2810669
2026-06   155 2856123
2026-07    91 1519185
2026-08    33  800800

[카드사별 집계]
카드사  거래건수    순사용액
 신한   124 4308543
 삼성   185 3053709
 KB   187 2397650

[카테고리별 집계]
  카테고리  거래건수    순사용액
    식비    74 2225835
    기타   157 2133764
 교통·차량    69 1443190
편의점·마트    86 1409663
 쇼핑·생활    12 1090040
 의료·보험    33  804420
 카페·간식    40  241230
주거·공과금    14  225960
교육·디지털     6  105900
    여가     5   79900

[순사용액 상위 가맹점 15개]
                  가맹점  거래건수   순사용액
             (주)넥슨코리아     7 896400
        (주)한섬 패션웨어하우스     1 672100
                 쿠팡이츠    23 657259
           주식회사 나눔에너지    12 613500
      쿠팡(쿠페이)-쿠팡(쿠페이)    16 574720
               마법의손의원     3 472000
                   쿠팡     8 365520
놀유니버스티켓-문화비-주식회사놀유니버스     1 292700
          주식회사 우아한형제들    12 225200
            

## 7. 카드 기록 조회·분석 RAG 챗봇

Qdrant는 질문과 의미적으로 가까운 근거 거래를 찾고, pandas는 질문 범위에 해당하는
전체 거래를 다시 필터링하여 정확한 건수와 합계를 계산합니다.

따라서 검색 결과 `k`개만 합산하는 오류 없이 카드사·월·카테고리·가맹점·취소 조건에
맞는 답변을 만들 수 있습니다.


In [91]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

template = """
당신은 개인 카드 사용 기록을 조회하고 소비 패턴을 분석하는 재무 도우미입니다.
반드시 아래 카드 기록과 pandas가 계산한 집계만 근거로 답하세요.

[답변 원칙]
1. 금액과 거래 건수는 반드시 [질문 범위의 정확한 집계] 값을 사용합니다.
2. 취소는 음수이며, 순사용액은 승인 금액과 취소 금액을 합한 값입니다.
3. 조건에 해당하는 기록이 없으면 없다고 명확히 말합니다.
4. 조회 질문에는 카드사, 이용일, 가맹점, 금액, 승인/취소 여부를 표로 정리합니다.
5. 분석 질문에는 핵심 수치, 패턴, 해석을 구분해 간결하게 답합니다.
6. 확인되지 않은 소비 목적이나 사용자의 의도를 추측하지 않습니다.
7. 답변 끝에 적용 조건과 데이터 기간을 표시합니다.

[적용된 조건]
{applied_filters}

[질문 범위의 정확한 집계]
{scope_summary}

[질문 범위의 정확한 거래 목록]
{exact_records}

[Qdrant 의미 검색 근거]
{retrieved_records}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=[
        "applied_filters", "scope_summary", "exact_records",
        "retrieved_records", "question",
    ],
    template=template,
)


def format_records(data: pd.DataFrame, limit: int = 200) -> str:
    if data.empty:
        return "해당 거래 없음"
    lines = []
    for _, row in data.sort_values(["이용일", "카드사"]).head(limit).iterrows():
        lines.append(
            f"- {row['이용일']:%Y-%m-%d} | {row['카드사']} | {row['가맹점']} | "
            f"{row['금액']:,}원 | {row['거래유형']} | {row['카테고리']}"
        )
    if len(data) > limit:
        lines.append(f"- 나머지 {len(data) - limit}건은 집계에는 포함되지만 목록에서는 생략")
    return "\n".join(lines)


def card_rag_chat(question: str) -> str:
    """의미 검색과 정확한 pandas 필터·집계를 결합해 답변합니다."""
    retrieved_docs = card_retriever.invoke(question)
    retrieved_records = "\n".join(
        f"- {doc.page_content} (출처: {doc.metadata['source']})"
        for doc in retrieved_docs
    ) or "관련 거래 없음"

    scope_data, applied = filter_transactions_for_question(question)
    has_scope = bool(applied)
    scope_summary = make_summary_tables(
        scope_data if has_scope else transactions,
        title="질문 범위" if has_scope else "전체",
    )
    exact_records = (
        format_records(scope_data)
        if has_scope
        else "특정 조건이 없어 전체 원장 대신 전체 집계와 의미 검색 결과를 사용합니다."
    )

    formatted_prompt = prompt_template.format(
        applied_filters=", ".join(applied) if applied else "특정 조건 없음",
        scope_summary=scope_summary,
        exact_records=exact_records,
        retrieved_records=retrieved_records,
        question=question,
    )
    return llm.invoke(formatted_prompt).content


print("✓ 카드 기록 조회·분석 RAG 챗봇 준비 완료")


✓ 카드 기록 조회·분석 RAG 챗봇 준비 완료


## 8. RAG 챗봇 테스트

In [92]:
questions = [
    "삼성카드 7월 교통비 기록과 합계를 보여줘",
    "카드사별 순사용액을 비교해줘",
    "6월 취소 거래를 모두 찾아서 순사용액에 미친 영향을 알려줘",
]

for question in questions:
    print(f"\n{'=' * 80}")
    print(f"질문: {question}")
    print(f"{'=' * 80}\n")
    display(Markdown(card_rag_chat(question)))



질문: 삼성카드 7월 교통비 기록과 합계를 보여줘



아래는 **2026-07월 삼성카드 교통·차량** 기록입니다.

| 카드사 | 이용일 | 가맹점 | 금액 | 승인/취소 |
|---|---:|---|---:|---|
| 삼성 | 2026-07-01 | 티머니택시(법인)_4 | 9,900원 | 승인 |
| 삼성 | 2026-07-11 | 코레일유통(주) | 1,100원 | 승인 |
| 삼성 | 2026-07-25 | 코레일유통(주) | 1,100원 | 승인 |
| 삼성 | 2026-07-29 | 코레일유통(주) | 1,100원 | 승인 |

### 합계
- 거래 건수: **4건**
- 승인 합계: **13,200원**
- 취소 합계: **0원**
- 순사용액(승인+취소): **13,200원**

### 적용 조건 및 데이터 기간
- 적용 조건: **월=2026-07, 카드사=삼성, 카테고리=교통·차량**
- 데이터 기간: **2026-07-01 ~ 2026-07-29**


질문: 카드사별 순사용액을 비교해줘



카드사별 순사용액은 아래와 같습니다.

| 카드사 | 거래 건수 | 순사용액 |
|---|---:|---:|
| 신한 | 124건 | 4,308,543원 |
| 삼성 | 185건 | 3,053,709원 |
| KB | 187건 | 2,397,650원 |

### 핵심 비교
- **신한카드**가 **4,308,543원**으로 가장 높습니다.
- **삼성카드**는 **3,053,709원**으로 2위입니다.
- **KB카드**는 **2,397,650원**으로 가장 낮습니다.

### 해석
- 신한카드의 순사용액은 삼성카드보다 **1,254,834원** 많고, KB카드보다 **1,910,893원** 많습니다.
- 거래 건수는 **KB(187건) > 삼성(185건) > 신한(124건)** 순이지만, 순사용액은 **신한 > 삼성 > KB** 순이어서, 건수와 사용액이 꼭 비례하지는 않습니다.

적용 조건: 특정 조건 없음  
데이터 기간: 2026-02-01 ~ 2026-08-19


질문: 6월 취소 거래를 모두 찾아서 순사용액에 미친 영향을 알려줘



6월 **취소 거래 14건**을 모두 반영한 결과, 순사용액은 **-148,960원**입니다.  
즉, 취소만 집계된 기간이므로 **승인 합계는 0원**, 순사용액은 전부 취소 금액의 합과 같습니다.

### 취소 거래 목록
| 카드사 | 이용일 | 가맹점 | 금액 | 승인/취소 |
|---|---|---:|---:|---|
| KB | 2026-06-02 | 포인트사용 | -2,300원 | 취소 |
| KB | 2026-06-02 | 포인트사용 | -5,600원 | 취소 |
| KB | 2026-06-03 | 포인트사용 | -51,200원 | 취소 |
| KB | 2026-06-08 | 포인트사용 | -6,550원 | 취소 |
| KB | 2026-06-09 | 포인트사용 | -11,900원 | 취소 |
| KB | 2026-06-10 | 포인트사용 | -2,300원 | 취소 |
| KB | 2026-06-11 | 포인트사용 | -48,640원 | 취소 |
| KB | 2026-06-11 | 포인트사용 | -560원 | 취소 |
| KB | 2026-06-11 | 포인트사용 | -350원 | 취소 |
| 삼성 | 2026-06-22 | 씨유(CU) 망포자이점 | -12,400원 | 취소 |
| KB | 2026-06-29 | 정상할인 | -900원 | 취소 |
| KB | 2026-06-29 | 정상할인 | -1,338원 | 취소 |
| KB | 2026-06-30 | 정상할인 | -2,932원 | 취소 |
| KB | 2026-06-30 | 배민클럽_우아한형제들 | -1,990원 | 취소 |

### 순사용액에 미친 영향
- **취소 합계:** -148,960원
- **승인 합계:** 0원
- **순사용액 변화:** **-148,960원 감소**

### 카드사별 영향
- **KB:** 13건, **-136,560원**
- **삼성:** 1건, **-12,400원**

### 핵심 해석
- 6월 취소 거래는 **KB 카드에 집중**되어 있습니다.
- 가맹점 기준으로는 **포인트사용** 취소가 가장 많아, **9건 / -129,400원**으로 순사용액 감소에 가장 큰 영향을 줬습니다.
- 그 밖에 **정상할인** 3건(-5,170원), **씨유(CU) 망포자이점** 1건(-12,400원), **배민클럽_우아한형제들** 1건(-1,990원)이 확인됩니다.

적용 조건: **월=2026-06, 거래유형=취소**  
데이터 기간: **2026-06-02 ~ 2026-06-30**

## 9. 자유 질문

아래 `question`만 바꿔 카드 기록을 조회하거나 분석할 수 있습니다.

### 질문 예시
- `6월 취소 거래를 모두 보여줘`
- `쿠팡에서 사용한 기록을 찾아줘`
- `고정비로 보이는 지출은 무엇이고 월평균은 어느 정도야?`
- `2월부터 7월까지 소비가 어떻게 변했어?`


In [95]:
question = "카드사 별 사용 총 금액"
answer = card_rag_chat(question)
display(Markdown(answer))


카드사별 사용 총 금액은 아래와 같습니다.  
※ 취소를 반영한 순사용액 기준입니다.

| 카드사 | 거래건수 | 사용 총 금액(순사용액) |
|---|---:|---:|
| 신한 | 124건 | 4,308,543원 |
| 삼성 | 185건 | 3,053,709원 |
| KB | 187건 | 2,397,650원 |

추가로, 전체 기간 합계는 9,759,902원입니다.

적용 조건: 특정 조건 없음  
데이터 기간: 2026-02-01 ~ 2026-08-19

## 프로젝트 완료 체크리스트

- [x] 카드 데이터 에셋 10개 전체 로딩(XLSX 6, HTML XLS 3, PDF 1)
- [x] 개인정보(카드번호·주민등록번호) 제외
- [x] 거래 단위 Document 및 카드사·날짜·금액 메타데이터 생성
- [x] Qdrant Cloud에 496개 거래 저장 및 고정 UUID로 중복 방지
- [x] 월·카드사·카테고리·거래유형 keyword 인덱스 생성
- [x] 의미 검색과 구조화 메타데이터 필터 결합
- [x] 질문 조건에 해당하는 전체 원장을 pandas로 정확하게 필터링
- [x] 승인·취소를 분리하고 순사용액 정확히 계산
- [x] 카드사별·월별·카테고리별·가맹점별 집계 제공
- [x] 검색 `k` 제한과 무관한 정확한 조회·분석 RAG 구현
- [x] 복합 조건 질문과 자유 질문 셀 제공

### 데이터 처리 원칙
- 카드사별 기록은 서로 다른 실제 거래이므로 파일 간 동일 날짜·가맹점·금액을 임의로 삭제하지 않습니다.
- 원본파일과 원본행을 결합한 고정 거래 ID로 재실행 시 Qdrant 중복을 방지합니다.
- 카테고리는 가맹점 키워드 규칙으로 일관되게 분류하며 규칙은 `classify_category()`에서 관리합니다.
- 새 데이터 파일은 형식별 로더가 자동 탐색하며, 추가 후 1번 섹션부터 다시 실행합니다.
- 임베딩 모델 변경 시 벡터 차원이 달라질 수 있으므로 새 컬렉션을 사용합니다.
